Load & Inspect Data

In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/ethiopia_fi_unified_data.csv")
ref = pd.read_csv("../data/raw/reference_codes.csv")

df.head()
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43 entries, 0 to 42
Data columns (total 34 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   record_id            43 non-null     object 
 1   record_type          43 non-null     object 
 2   category             10 non-null     object 
 3   pillar               33 non-null     object 
 4   indicator            43 non-null     object 
 5   indicator_code       43 non-null     object 
 6   indicator_direction  33 non-null     object 
 7   value_numeric        33 non-null     float64
 8   value_text           10 non-null     object 
 9   value_type           43 non-null     object 
 10  unit                 33 non-null     object 
 11  observation_date     43 non-null     object 
 12  period_start         10 non-null     object 
 13  period_end           10 non-null     object 
 14  fiscal_year          43 non-null     object 
 15  gender               43 non-null     objec

Record Type Counts

In [2]:
df['record_type'].value_counts()


record_type
observation    30
event          10
target          3
Name: count, dtype: int64

Temporal Coverage

In [3]:
df['observation_date'] = pd.to_datetime(df['observation_date'], errors='coerce')
df.groupby(df['observation_date'].dt.year)['indicator_code'].count()


observation_date
2014     1
2017     1
2021     7
2022     1
2023     2
2024    14
2025    15
2028     1
2030     1
Name: indicator_code, dtype: int64

Indicators Coverage

In [4]:
df.groupby('indicator_code')['observation_date'].count().sort_values()


indicator_code
ACC_MOBILE_PEN        1
EVT_ETHIOPAY          1
EVT_CROSSOVER         1
AFF_DATA_INCOME       1
EVT_MPESA_INTEROP     1
EVT_MPESA             1
EVT_FX_REFORM         1
EVT_FAYDA             1
EVT_NFIS2             1
EVT_SAFARICOM         1
EVT_SAFCOM_PRICE      1
EVT_TELEBIRR          1
USG_ATM_COUNT         1
USG_ATM_VALUE         1
USG_ACTIVE_RATE       1
GEN_GAP_MOBILE        1
USG_TELEBIRR_USERS    1
USG_TELEBIRR_VALUE    1
USG_P2P_VALUE         1
USG_MPESA_USERS       1
USG_CROSSOVER         1
USG_MPESA_ACTIVE      1
ACC_4G_COV            2
ACC_MM_ACCOUNT        2
USG_P2P_COUNT         2
GEN_MM_SHARE          2
GEN_GAP_ACC           2
ACC_FAYDA             4
ACC_OWNERSHIP         7
Name: observation_date, dtype: int64

Events Overview

In [6]:
events = df[df['record_type'] == 'event']

events[['indicator', 'category', 'observation_date']] \
    .rename(columns={
        'indicator': 'event_name',
        'observation_date': 'event_date'
    }) \
    .sort_values('event_date')


,event_name,category,event_date
33,Telebirr Launch,product_launch,2021-05-17
41,NFIS-II Strategy Launch,policy,2021-09-01
34,Safaricom Ethiopia Commercial Launch,market_entry,2022-08-01
35,M-Pesa Ethiopia Launch,product_launch,2023-08-01
36,Fayda Digital ID Program Rollout,infrastructure,2024-01-01
37,Foreign Exchange Liberalization,policy,2024-07-29
38,P2P Transaction Count Surpasses ATM,milestone,2024-10-01
39,M-Pesa EthSwitch Integration,partnership,2025-10-27
42,Safaricom Ethiopia Price Increase,pricing,2025-12-15
40,EthioPay Instant Payment System Launch,infrastructure,2025-12-18


DATA ENRICHMENT (NEW RECORDS)

A. New Observation (Smartphone penetration)

In [7]:
new_obs = [
    {
        "record_type": "observation",
        "pillar": "usage",
        "indicator": "Smartphone penetration",
        "indicator_code": "smartphone_penetration",
        "value_numeric": 44,
        "observation_date": "2023-01-01",
        "source_name": "GSMA",
        "source_url": "https://www.gsma.com",
        "confidence": "medium",
        "original_text": "Smartphone adoption in Ethiopia reached 44% in 2023",
        "collected_by": "Kalkidan Abreham",
        "collection_date": "2026-01-30",
        "notes": "Key enabler for digital payments"
    }
]

df = pd.concat([df, pd.DataFrame(new_obs)], ignore_index=True)


B. New Event

In [8]:
new_events = [
    {
        "record_type": "event",
        "indicator": "Telebirr merchant QR expansion",
        "indicator_code": "telebirr_qr_expansion",
        "category": "infrastructure",
        "observation_date": "2022-06-01",
        "source_name": "EthSwitch",
        "source_url": "https://www.ethswitch.com",
        "confidence": "medium",
        "original_text": "Nationwide rollout of QR-based merchant payments",
        "notes": "Expected to increase usage rather than access"
    }
]

df = pd.concat([df, pd.DataFrame(new_events)], ignore_index=True)


C. Impact Links

In [9]:
impact_links = [
    {
        "record_type": "impact_link",
        "parent_id": "telebirr_qr_expansion",
        "pillar": "usage",
        "related_indicator": "digital_payment_adoption",
        "impact_direction": "positive",
        "impact_magnitude": "medium",
        "lag_months": 6,
        "evidence_basis": "Comparable QR rollouts in Kenya & India"
    }
]

df = pd.concat([df, pd.DataFrame(impact_links)], ignore_index=True)


D. Save Enriched Dataset

In [11]:
df.to_csv("../data/processed/ethiopia_fi_enriched.csv", index=False)


Data Enrichment Rationale
Smartphone penetration and merchant QR infrastructure are added as enabling indicators for digital payment usage. Events are recorded without pre-assigning pillars, and their effects are modeled separately using impact_link records, in line with the unified schema design.